# DINO-Tracker: Liver Ultrasound Video Tracking

This notebook runs the DINO-Tracker pipeline on liver ultrasound videos.

**Requirements**: Select a GPU runtime (Runtime → Change runtime type → T4 GPU)

## 1. Clone the Repository

In [ ]:
import os
if not os.path.exists('/content/dinotracker-testing'):
    !git clone https://github.com/rafbacq/dinotracker-testing.git /content/dinotracker-testing
os.chdir('/content/dinotracker-testing')
print(f'Working directory: {os.getcwd()}')

## 2. Install Dependencies

This fixes the numpy/OpenCV version conflict on Colab.

In [ ]:
# Fix numpy/OpenCV incompatibility (Colab has numpy 2.x, OpenCV needs 1.x)
!pip install --quiet "numpy<2" "opencv-python>=4.9"

# Install DINO-Tracker dependencies
!pip install --quiet einops imageio imageio-ffmpeg kornia matplotlib mediapy \
    pandas pillow tqdm PyYAML antialiased_cnns

# Install xformers (may show warnings, that's OK)
!pip install --quiet xformers 2>/dev/null || echo 'xformers install warning (usually OK)'

# Verify critical imports
import numpy; print(f'numpy:  {numpy.__version__}')
import cv2;   print(f'opencv: {cv2.__version__}')
import torch;  print(f'torch:  {torch.__version__}')
print(f'CUDA:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')

## 3. Set Environment

In [ ]:
import sys
project_root = '/content/dinotracker-testing'
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.environ['PYTHONPATH'] = f"{project_root}:{os.environ.get('PYTHONPATH', '')}"
print(f'PYTHONPATH set to include {project_root}')

## 4. Run the Full Pipeline

This processes all 7 ultrasound videos through:
1. Frame extraction (grayscale → RGB)
2. Preprocessing (RAFT optical flow, DINO embeddings)
3. Training (self-supervised, per-video)
4. Inference (grid-based trajectory prediction)
5. Visualization (tracked points overlaid on video)

**Estimated time**: 4-9 hours for all 7 videos on T4 GPU.

To test with just one video first, change `--videos` below.

In [ ]:
# Process ALL videos (takes several hours)
!cd /content/dinotracker-testing && \
  PYTHONPATH=/content/dinotracker-testing:$PYTHONPATH \
  python run_ultrasound_pipeline.py \
    --input-dir usliverseq-mp4 \
    --output-dir outputs \
    --target-frames 150 \
    --vis-fps 10

In [ ]:
# --- OR: Process just ONE video for a quick test ---
# !cd /content/dinotracker-testing && \
#   PYTHONPATH=/content/dinotracker-testing:$PYTHONPATH \
#   python run_ultrasound_pipeline.py \
#     --input-dir usliverseq-mp4 \
#     --output-dir outputs \
#     --target-frames 150 \
#     --videos volunteer01.mp4

## 5. View Results

In [ ]:
# Check pipeline summary
import json
if os.path.exists('outputs/pipeline_summary.json'):
    with open('outputs/pipeline_summary.json') as f:
        summary = json.load(f)
    for r in summary['results']:
        steps = ', '.join(f"{k}: {v}" for k, v in r['steps'].items())
        print(f"{r['video']}: {steps}")

In [ ]:
# Display a visualization video (if available)
from IPython.display import HTML
from base64 import b64encode
import glob

vis_videos = sorted(glob.glob('outputs/*/visualizations/*.mp4'))
if vis_videos:
    print(f'Found {len(vis_videos)} visualization(s):')
    for v in vis_videos:
        print(f'  {v}')
    # Display the first one
    with open(vis_videos[0], 'rb') as f:
        video_data = f.read()
    b64 = b64encode(video_data).decode()
    display(HTML(f'<video controls width="640"><source src="data:video/mp4;base64,{b64}"></video>'))
else:
    print('No visualization videos found yet. Pipeline may still be running.')

In [ ]:
# List output structure
!find outputs -maxdepth 3 -type d | head -50